### **Vision-Language Medical Foundation Models**

#### **1.3. Zero-shot classification**
---

Pre-trained vision-language models are capable or performing the so-called **zero-shot predictions**. These image-level predictions are driven by the language encoder, thanks to the multi-modal alignment. **Given a set of descriptions for a subset of target categories, we can compute text prototypes of each category** in the common space. Given a new image, **the class assigned will be the one corresponding to the most similar text prototype** to the image embedding.

In this notebook, we will explore two popular forms to perform zero-shot predictions: single prompt, and prompt ensemble.


In [ ]:
# General imports
import warnings
warnings.filterwarnings('ignore')

import copy
from pathlib import Path
import shutil

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

# Device for training/inference
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Available device: " + device)


#### **Dataset details**

In [ ]:
# SICAPv2 dataset setup
# The shared copy is stored in the project folder. Each user copies it once to local_data.
SHARED_DATA_ROOT = Path.home() / "projects" / "def-sponsor00" / "SICAPv2"
LOCAL_DATA_ROOT = Path.home() / "local_data" / "datasets" / "SICAPv2"

if not LOCAL_DATA_ROOT.exists():
    LOCAL_DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
    print("Copying SICAPv2 from shared project folder to local_data...")
    shutil.copytree(SHARED_DATA_ROOT, LOCAL_DATA_ROOT)
else:
    print("SICAPv2 already exists in local_data.")

DATA_ROOT = LOCAL_DATA_ROOT
IMAGE_DIR = DATA_ROOT / "images"

categories = ["NC", "G3", "G4", "G5"]  # List of categories

path_images = str(IMAGE_DIR) + "/"
dataframe_train = str(DATA_ROOT / "partition" / "Test" / "Train.xlsx")
dataframe_test = str(DATA_ROOT / "partition" / "Test" / "Test.xlsx")

# Quick sanity checks
assert IMAGE_DIR.exists(), f"Image folder not found: {IMAGE_DIR}"
assert Path(dataframe_train).exists(), f"Train spreadsheet not found: {dataframe_train}"
assert Path(dataframe_test).exists(), f"Test spreadsheet not found: {dataframe_test}"

print("Dataset ready at:", DATA_ROOT)
print("Images:", IMAGE_DIR)
print("Test labels:", dataframe_test)


#### **Load model, pre-processing, and wrapper**

In [ ]:
# Load model and pre-processing tools from HuggingFace
from transformers import AutoProcessor, AutoModel

# In Transformers, a model is loaded by its repository/model ID.
# For PLIP, the ID is "vinid/plip".
processor = AutoProcessor.from_pretrained("vinid/plip")  # image/text pre-processing
processor.image_processor.do_center_crop = False

plip = AutoModel.from_pretrained("vinid/plip").eval().to(device)  # model with pre-trained weights
# eval() avoids dropout and training-mode behavior during inference.


In [ ]:
# Again, we will use our PLIP Wrapper for easy interaction
class PLIPWrapper(torch.nn.Module):
    def __init__(self, encoder, proj_layer):
        super().__init__()

        self.encoder = encoder         # One-modality encoder from VLM.
        self.proj_layer = proj_layer   # Projection layer into joint embedding space.

    def forward(self, inputs):
        # Forward input through encoder
        features = self.encoder(**inputs).pooler_output  # global image/text feature

        # Project features into joint image-text space
        projected_features = self.proj_layer(features)

        # L2-normalize features for cosine-similarity classification
        projected_features = projected_features / projected_features.norm(dim=-1, keepdim=True)

        return projected_features

# Create model wrappers for vision and text encoders
vision_encoder = PLIPWrapper(plip.vision_model, plip.visual_projection).to(device).eval()
text_encoder = PLIPWrapper(plip.text_model, plip.text_projection).to(device).eval()


#### **Feature extraction**
First, we need to extract all feature representations from the test subset.

In [ ]:
print(processor.image_processor.crop_size)
print(type(processor.image_processor.crop_size))

In [ ]:
# To run it on the whole dataset, we move the vision model to the selected device.
vision_encoder.to(device).eval()

# We need to format the pre-processing transforms in a friendly format for torch DataLoaders.
from torchvision import transforms

# Pre-processing transforms to apply during data loading.
# We keep it explicit and version-stable instead of relying on processor.image_processor.crop_size internals.
plip_transforms = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to PLIP/CLIP resolution
    transforms.ToTensor(),          # Move PIL image to tensor [C,H,W]
    transforms.Normalize(
        mean=processor.image_processor.image_mean,
        std=processor.image_processor.image_std
    )
])


In [ ]:
# Create a self-contained SICAPv2 dataset and DataLoader.
# This replaces the original hidden helper: from vlms.data import loader

from torch.utils.data import Dataset, DataLoader

class SICAPv2ClassificationDataset(Dataset):
    def __init__(self, dataframe_path, path_images, categories, transforms=None):
        self.data = pd.read_excel(dataframe_path)
        self.path_images = Path(path_images)
        self.categories = categories
        self.transforms = transforms

        # Keep only rows with a valid image name and at least one class indicator.
        self.data = self.data.dropna(subset=["image_name"]).reset_index(drop=True)
        self.data = self.data[self.data[categories].sum(axis=1) > 0].reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image_path = self.path_images / row["image_name"]
        image = Image.open(image_path).convert("RGB")

        if self.transforms is not None:
            image = self.transforms(image)

        # The spreadsheet has one-hot / multi-hot columns NC, G3, G4, G5.
        # For classification, we use the strongest/active class.
        label = int(np.argmax(row[self.categories].to_numpy(dtype=np.float32)))

        return image, label, str(image_path)


def loader(dataframe_path, path_images, categories, transforms, batch_size=8, num_workers=0, shuffle=False):
    dataset = SICAPv2ClassificationDataset(
        dataframe_path=dataframe_path,
        path_images=path_images,
        categories=categories,
        transforms=transforms
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available()
    )


test_loader = loader(
    dataframe_path=dataframe_test,
    path_images=path_images,
    categories=categories,
    transforms=plip_transforms,
    batch_size=8,
    num_workers=0
)

# We can check the dataset format and available samples
print("Samples available for testing:", len(test_loader.dataset.data))
print(test_loader.dataset.data.iloc[0])


In [ ]:
# Extract image features from the frozen PLIP vision encoder.
# This replaces the original hidden helper: from vlms.utils import extract_features

def extract_features(data_loader, vision_encoder):
    vision_encoder.eval()
    all_features = []
    all_labels = []

    with torch.no_grad():
        for images, labels, _paths in tqdm(data_loader, desc="Extracting image features"):
            inputs = {"pixel_values": images.to(device)}
            features = vision_encoder(inputs)

            all_features.append(features.detach().cpu())
            all_labels.append(labels.detach().cpu())

    X = torch.cat(all_features, dim=0).numpy()
    Y = torch.cat(all_labels, dim=0).numpy()
    return X, Y


X_test, Y_test = extract_features(test_loader, vision_encoder)

# Let's check the test features
print("Test features:", X_test.shape)
print("Test labels:", Y_test.shape)


#### **1.3.1. Single prompt**

One the vision-language is pre-trained, **vision and image feature spaces are aligned**. Thats is to say: a **text description of a concept should produce a similar representation in the shared embedding space than an image of such category**. This phenomenon is leveraged to perform **zero-shot prediction without any adaptation** of the VLM. **Class-wise embeddings (class prototypes)** are computed from text descriptions of each category. Such prototypes for $C$ categories and $D_t$ features can be embeded into a feature matrix $W$. **Note that this is similar to a Linear layer in a classical MLP output (without bias term)**. Thus, **class predictions (logits) can be computed fron vision features $F_v$ by performing matrix multiplication**, as in a fully-connected layer: $F_v \times W^T$  = out -> (1xDv) x (DvxC) = (1xC)

In [ ]:
# Define text input
class_prompts = ["non-cancerous",
                 "Gleason grade 3",
                 "Gleason grade 4",
                 "Gleason grade 5"]

# Tokenize text
inputs = processor.tokenizer(
    class_prompts,
    max_length=77,
    padding=True,
    truncation=True,
    return_tensors="pt"
)
inputs = {k: v.to(device) for k, v in inputs.items()}

# Compute text prototypes per class
text_encoder.to(device).eval()
with torch.no_grad():
    class_prototypes = text_encoder(inputs)


In [ ]:
# Compute predictions
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32, device=device)
    prob = torch.softmax(
        X_test_tensor @ class_prototypes.t() * plip.logit_scale.exp(),
        dim=-1
    )
    prob = prob.detach().cpu().numpy()

print("Prediction shape:", prob.shape)
print("Example:", prob[0, :])


In [ ]:
# Compute metrics
# This replaces the original hidden helper: from vlms.utils import evaluate

from sklearn.metrics import balanced_accuracy_score, confusion_matrix

def evaluate(y_true, prob):
    y_pred = np.argmax(prob, axis=1)
    aca = balanced_accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(categories))))
    return aca, cm

aca, cm = evaluate(Y_test, prob)
print("Balanced accuracy:", aca)
print("Confusion matrix:")
print(cm)


#### **1.3.2. Prompt ensemble**

A popular option to refine the text prototype, is to **combine multiple prompts (prompt ensemble)**, which are averaged per class. Thus, **noisy features in an specific prompt are alleviated**, and usually, performance is improved. Such prompt ensemble comes usually from using **different templates**, i.e. "A photo of [CLS]", and **different descriptions** of the target class.

In [ ]:
# Ensemble of templates
templates = ["a histopathology slide showing [CLS]", "histopathology image of [CLS]",
             "pathology tissue showing [CLS]", "presence of [CLS] tissue on image"]

# Category-wise descriptions, which are more informative than category names. For instance, "atrophic dense glands" better 
# describes the local findings associated with Gleason grade 3.
prompts_dict = {"NC": ["benign glands"],
                "G3": ["atrophic dense glands"],
                "G4": ["cribriform ill-formed fused papillary patterns"],
                "G5": ["isolated nest cells without lumen roseting patterns"]}

# Combine all paired options of templates and descriptions
prompts = {}
for iCategory in categories:
    prompts[iCategory] = [caption.replace("[CLS]", iDescription) for iDescription in prompts_dict[iCategory]
                          for caption in templates]

# Compute embeddings per category
class_prototypes = []
text_encoder.to(device).eval()
for iKey in range(len(categories)):
    with torch.no_grad():
        # Retrieve descriptions for that particular category
        descriptions = prompts[categories[iKey]]
        # Tokenize text
        inputs = processor.tokenizer(
            descriptions,
            max_length=77,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Forward text encoder
        text_features_ensemble = text_encoder(inputs)
        # Get class prototypes as average of all text prompts
        avg_text_features = text_features_ensemble.mean(0).unsqueeze(0)
        # Re-normalize embedding
        avg_text_features = avg_text_features / avg_text_features.norm(dim=-1, keepdim=True)
        class_prototypes.append(avg_text_features)

# Concatenate all class prototypes
class_prototypes = torch.cat(class_prototypes, dim=0)


In [ ]:
# Compute predictions
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32, device=device)
    prob = torch.softmax(
        X_test_tensor @ class_prototypes.t() * plip.logit_scale.exp(),
        dim=-1
    )
    prob = prob.detach().cpu().numpy()

print("Prediction shape:", prob.shape)
print("Example:", prob[0, :])


In [ ]:
# Compute metrics
aca, cm = evaluate(Y_test, prob)
print("Balanced accuracy:", aca)
print("Confusion matrix:")
print(cm)
# As you can see, prompt ensembling can boost the performance.


--- 
##### **ACTIVITY**

Well, now you know everything you need about zero-shot predictions. If you want to know more, I recommend:

- Try developing the same pipeline for [CONCH](https://huggingface.co/MahmoodLab/CONCH) [1], a recently introduced VLM for histology. Its vision backbone is large scale, takes large-resolution input images, and is pre-trained with more data. How does model scaling translates to black-box Adaptation?
- Indeed, CONCH also uses a different subset of templates for zero-shot in Gleason grades (which I share below). How the prompt selection affect performance? How realistic is designing prompts to optimize test performance?


In [ ]:
# Text ensemble for CONCH
templates = ["[CLS].",
            "a photomicrograph showing [CLS].",
            "a photomicrograph of [CLS].",
            "an image of [CLS].",
            "an image showing [CLS].",
            "an example of [CLS].",
            "[CLS] is shown.",
            "this is [CLS].",
            "there is [CLS].",
            "a histopathological image showing [CLS].",
            "a histopathological image of [CLS].",
            "a histopathological photograph of [CLS].",
            "a histopathological photograph showing [CLS].",
            "shows [CLS].",
            "presence of [CLS].",
            "[CLS] is present.",
            "an H&E stained image of [CLS].",
            "an H&E stained image showing [CLS].",
            "an H&E image showing [CLS].",
            "an H&E image of [CLS].",
            "[CLS], H&E stain.",
            "[CLS], H&E."]

prompts_dict = {"NC": ["non-cancerous tissue", "non-cancerous prostate tissue", "benign tissue", "benign glands", 
                       "benign prostate glands", "benign prostate tissue"],
                "G3": ["gleason grade 3", "gleason pattern 3", "prostate cancer, gleason grade 3", 
                       "prostate cancer, gleason pattern 3", "prostate adenocarcinoma, well-differentiated",
                       "well-differentiated prostatic adenocarcinoma"],
                "G4": ["gleason grade 4", "gleason pattern 4", "prostate cancer, gleason grade 4", 
                       "prostate cancer, gleason pattern 4", "prostate adenocarcinoma, moderately differentiated",  
                       "moderately differentiated prostatic adenocarcinoma"],
                "G5": ["gleason grade 5", "gleason pattern 5", "prostate cancer, gleason grade 5",
                       "prostate cancer, gleason pattern 5", "prostate adenocarcinoma, poorly differentiated",
                       "poorly differentiated prostatic adenocarcinoma"]}


# These CONCH templates are included as optional examples for discussion.
# They are not executed in this PLIP zero-shot pipeline unless reused above. 

--- 
## **References**

[1] Lu, M.Y., Chen, B., Williamson, D.F.K. et al. (2024) A visual-language foundation model for computational pathology. Nature Medicine.

--- 